# Colab Test Notebook - Product Detection Pipeline

A self-contained checker to run the pipeline in **Google Colab**.

It verifies each stage step by step:
1. Install deps (Colab already ships CUDA torch)
2. Bring in the project files (`config.yaml`, `pipeline_utils.py`, `product_prompts.txt`)
3. Set secrets (SerpApi key, AWS creds) via Colab
4. Load config + confirm the product prompts file loads
5. Provide a video (upload / URL / S3)
6. Run detection -> dedup -> (optional) S3 -> match -> metadata

> **GPU:** Runtime -> Change runtime type -> Hardware accelerator: **GPU**.

## 1. Install dependencies
Colab already has a CUDA build of torch, so we skip reinstalling it.

In [ ]:
!pip install -q ultralytics open-clip-torch opencv-python-headless imagehash \
  yt-dlp boto3 google-search-results pyyaml python-dotenv tqdm

## 2. Get the project files into Colab
Pick **ONE** option and run it.

You need `pipeline_utils.py`, `config.yaml`, and `product_prompts.txt` in the working dir.

In [ ]:
# OPTION A - clone from your GitHub repo (recommended).
# Replace the URL with your repo, then this puts you inside it.
REPO_URL = 'https://github.com/your-org/isteam_object_detection.git'

import os
if not os.path.exists('isteam_object_detection'):
    !git clone $REPO_URL
%cd isteam_object_detection
!ls -la

In [ ]:
# OPTION B - upload the three files by hand instead of cloning.
# Run this ONLY if you did not clone in Option A.
# from google.colab import files
# print('Upload pipeline_utils.py, config.yaml, product_prompts.txt')
# files.upload()

## 3. Secrets (SerpApi key + AWS creds)

Preferred: store them in Colab **Secrets** (key icon in the left sidebar) with these names:
`SERPAPI_API_KEY`, `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION`.
This cell reads them and exports them as environment variables.

In [ ]:
import os

def _load_secret(name):
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(name)

for _k in ['SERPAPI_API_KEY', 'AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY',
           'AWS_DEFAULT_REGION']:
    _v = _load_secret(_k)
    if _v:
        os.environ[_k] = _v

os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')
print('SERPAPI set:', bool(os.environ.get('SERPAPI_API_KEY')))
print('AWS creds set:', bool(os.environ.get('AWS_ACCESS_KEY_ID')))
print('Region:', os.environ.get('AWS_DEFAULT_REGION'))

## 4. Load config + confirm the product prompts file
This checks that `product_prompts.txt` loads and reports how many product names it found.

In [ ]:
import importlib, pipeline_utils as pu
importlib.reload(pu)

cfg = pu.load_config('config.yaml')
prompts = pu.load_product_prompts(cfg)
print('Prompts file :', pu.get(cfg, 'detection.product_prompts_file'))
print('Prompt count :', len(prompts))
print('First 15     :', prompts[:15])
assert len(prompts) > 50, 'Expected a large prompt list - is product_prompts.txt present?'

import torch
print('\nCUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Match threshold:', pu.get(cfg, 'matching.min_match_score'))
print('S3 public_read :', pu.get(cfg, 's3.public_read'))

## 5. Provide a video
Pick ONE. Each option sets `input.source_type` so the rest of the pipeline just works.

In [ ]:
# OPTION A - upload a local video file from your computer.
from google.colab import files
uploaded = files.upload()
video_name = list(uploaded.keys())[0]
cfg['input']['source_type'] = 'local'
cfg['input']['local_path'] = video_name
print('Using uploaded video:', video_name)

In [ ]:
# OPTION B - use a public URL (uncomment).
# cfg['input']['source_type'] = 'url'
# cfg['input']['url'] = 'https://example.com/video.mp4'

# OPTION C - use a public S3 link (uncomment).
# cfg['input']['source_type'] = 's3'
# cfg['input']['s3_uri'] = 's3://my-public-bucket/videos/input.mp4'

## 6. Run the pipeline (staged, with checks)

In [ ]:
from tqdm.auto import tqdm
from collections import Counter

# --- ingest + frames ---
video_path = pu.ingest_video(cfg)
video_id = pu.slugify(video_path.stem)
frames = pu.sample_frames(video_path, cfg)
print(f'Video: {video_path}  | id={video_id}  | frames sampled={len(frames)}')
assert frames, 'No frames sampled - check the video / time window settings.'

In [ ]:
# --- detection (humans ignored) ---
detector = pu.Detector(cfg)
crops_dir = pu.get(cfg, 's3.local_crops_dir', 'output/crops')
all_detections = []
for fr in tqdm(frames, desc='Detecting'):
    for d in detector.detect_frame(fr):
        detector.save_crop(fr, d, crops_dir)
        all_detections.append(d)
print(f'{len(all_detections)} detections (humans filtered).')
print('By label:', dict(Counter(d.label for d in all_detections)))

In [ ]:
# --- dedup into distinct products ---
embedder = pu.Embedder(cfg, section='dedup')
embeddings = embedder.embed_image_paths([d.crop_path for d in all_detections])
products = pu.dedup_products(all_detections, embeddings, cfg)
print(f'{len(products)} distinct products.')
for p in products:
    print(f'  {p.product_id}: {p.label}  {p.first_seen:.1f}s->{p.last_seen:.1f}s  '
          f'({len(p.occurrences)} occ.)')

In [ ]:
# --- preview the distinct crops ---
import matplotlib.pyplot as plt
from PIL import Image
n = len(products)
if n:
    cols = min(4, n); rows = (n + cols - 1) // cols
    plt.figure(figsize=(cols*3, rows*3))
    for i, p in enumerate(products):
        plt.subplot(rows, cols, i+1)
        plt.imshow(Image.open(p.representative_crop))
        plt.title(f'{p.product_id}: {p.label}', fontsize=9); plt.axis('off')
    plt.tight_layout(); plt.show()

In [ ]:
# --- upload crops to S3 BEFORE matching, so Google Lens can fetch them by URL ---
# By default objects stay private and we return a presigned URL (works on any
# bucket). Set s3.public_read: true in config.yaml only if your bucket allows ACLs.
uploader = pu.S3Uploader(cfg, video_id=video_id)
if uploader.enabled:
    uploader.preflight()   # fails fast with a clear message if bucket/creds are wrong
    for p in tqdm(products, desc='S3 upload'):
        suffix = f'{p.product_id}_{pu.slugify(p.label)}.jpg'
        p.s3_url = uploader.upload(p.representative_crop, key_suffix=suffix)
    print('Example fetchable URL:', products[0].s3_url[:120] if products else '(none)')
else:
    print('S3 disabled in config (s3.enabled: false) - skipping upload.')
    print('NOTE: Google Lens needs a fetchable image URL, so matching will be skipped.')

In [ ]:
# --- match to ecommerce products (threshold + trusted domains applied) ---
matcher = pu.Matcher(cfg, embedder=embedder)
for p in tqdm(products, desc='Matching'):
    try:
        p.recommendations = matcher.match(p, image_url=p.s3_url or None)
    except Exception as e:
        print(f'[warn] {p.product_id} ({p.label}): {e}')
        p.recommendations = []

thr = pu.get(cfg, 'matching.min_match_score')
for p in products:
    print(f'\n{p.product_id} [{p.label}] -> {len(p.recommendations)} matches >= {thr}')
    for r in p.recommendations:
        print(f'   {r.score:.2f}  {r.title[:55]!r}  {r.price}  {r.source}')
        print(f'         {r.url}')

In [ ]:
# --- write timestamped metadata (JSON + WebVTT) ---
import cv2, json
cap = cv2.VideoCapture(str(video_path))
fps = cap.get(cv2.CAP_PROP_FPS); fc = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
video_info = {
    'id': video_id, 'path': str(video_path), 'fps': fps, 'frame_count': fc,
    'width': int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0),
    'height': int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0),
    'duration_seconds': (fc/fps) if fps else None,
}
cap.release()
payload = pu.write_metadata(products, video_info, cfg)
print('Wrote', pu.get(cfg, 'metadata.output_path'), 'and', pu.get(cfg, 'metadata.webvtt_path'))
print(json.dumps(payload, indent=2)[:2000])

In [ ]:
# --- download the results out of Colab ---
from google.colab import files
try:
    files.download(pu.get(cfg, 'metadata.output_path'))
    files.download(pu.get(cfg, 'metadata.webvtt_path'))
except Exception as e:
    print('Download skipped:', e)

---
**Tuning is all in `config.yaml`:** `matching.min_match_score` (the 0.90), `detection.confidence_threshold`,
`frames.sample_every_seconds`, `dedup.same_product_similarity`, `matching.trusted_domains`.
Add products by editing **`product_prompts.txt`** and re-running from step 4.